# Travelling tracks: group assignment

**Group:** BDS-RGK3 · **Members:** Martin Davidsen

The first two questions are in
[assignment.md](../courses/ds4b-m1-3-pandas-2026/assignment.md), and also on the Moodle page.

This notebook currently covers Questions 1 and 2. Work through it from top to bottom: run the setup, complete each code section, and then read the answer block underneath.

Before you hand it in: **Restart & Run All**. If it does not run from start to finish in a clean
kernel, it is not ready.


In [21]:
# Setup. This cell is given to you; you should not need to change it.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print(f"pandas {pd.__version__}")
print(f"numpy {np.__version__}")

# The two data files. These URLs work anywhere, Colab included.
SONGS_URL = "https://raw.githubusercontent.com/aaubs/ds-master/codex/m1-pandas-2026/data/M1_2026/spotify_songs.csv"
FAMILIES_URL = "https://raw.githubusercontent.com/aaubs/ds-master/codex/m1-pandas-2026/data/M1_2026/subgenre_families.csv"


def load_class_csv(filename, url):
    """Read a class data file: the local copy if there is one, otherwise the URL above."""
    candidates = [Path("data/M1_2026") / filename,
                  Path("../data/M1_2026") / filename,
                  Path("ds-master/data/M1_2026") / filename,
                  Path("../ds-master/data/M1_2026") / filename]
    for path in candidates:
        if path.exists():
            print(f"Loaded {filename} from {path}")
            return pd.read_csv(path)
    print(f"Loaded {filename} from {url}")
    return pd.read_csv(url)


songs_raw = load_class_csv("spotify_songs.csv", SONGS_URL)
families_raw = load_class_csv("subgenre_families.csv", FAMILIES_URL)

print(f"songs_raw:    {songs_raw.shape[0]:,} rows x {songs_raw.shape[1]} columns")
print(f"families_raw: {families_raw.shape[0]:,} rows x {families_raw.shape[1]} columns")


pandas 3.0.5
numpy 2.4.6
Loaded spotify_songs.csv from https://raw.githubusercontent.com/aaubs/ds-master/codex/m1-pandas-2026/data/M1_2026/spotify_songs.csv
Loaded subgenre_families.csv from https://raw.githubusercontent.com/aaubs/ds-master/codex/m1-pandas-2026/data/M1_2026/subgenre_families.csv
songs_raw:    32,833 rows x 23 columns
families_raw: 24 rows x 2 columns


## Question 1 · What is one row, and how far does each track travel?


In [22]:
# What one row of songs_raw is. Show the output, not just the claim.

track_summary = (songs_raw.groupby("track_id", as_index=False)
                .agg(track_name=("track_name", "first"),
                     track_popularity=("track_popularity", "first"),
                     release_date=("track_album_release_date", "first"),
                     playlist_count=("playlist_id", "nunique")))

# Show several raw playlist associations for one track.
example_track_id = songs_raw["track_id"].iloc[0]
raw_example = songs_raw.loc[
    songs_raw["track_id"] == example_track_id,
    ["track_id", "playlist_id", "playlist_name", "playlist_genre", "playlist_subgenre"]
].drop_duplicates()
display(raw_example)

# Show the resulting one-row-per-track record.
display(track_summary.loc[track_summary["track_id"] == example_track_id])

# Checks: one summary row per track, and playlist_count is distinct playlists.
assert track_summary["track_id"].is_unique
expected_count = songs_raw.loc[
    songs_raw["track_id"] == example_track_id, "playlist_id"
].nunique()
actual_count = int(track_summary.loc[
    track_summary["track_id"] == example_track_id, "playlist_count"
].iloc[0])
assert actual_count == expected_count
print(f"Raw rows for this track: {len(raw_example)}")
print(f"Distinct playlists counted: {actual_count}")

,track_id,playlist_id,playlist_name,playlist_genre,playlist_subgenre
0,6f807x0ima9a1j3VPbc7VN,37i9dQZF1DXcZDD7cfEKhW,Pop Remix,pop,dance pop
29684,6f807x0ima9a1j3VPbc7VN,4aUEH3uhbofktrFkXOOaKj,Pop EDM Remixes,edm,pop edm


,track_id,track_name,track_popularity,release_date,playlist_count
24150,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,66,2019-06-14,2


Raw rows for this track: 2
Distinct playlists counted: 2


In [23]:
# Create a defensible release year from the observed year prefix.
track_summary["release_year"] = pd.to_numeric(
    track_summary["release_date"].astype("string").str[:4],
    errors="coerce")
display(track_summary.head())

,track_id,track_name,track_popularity,release_date,playlist_count,release_year
0,0017A6SJgTbfQVU2EtsPNo,Pangarap,41,2001-01-01,1,2001
1,002xjHwzEx66OWFV2IP9dk,The Others,15,2018-01-26,1,2018
2,004s3t0ONYlzxII9PLgU6z,I Feel Alive,28,2017-11-21,1,2017
3,008MceT31RotUANsKuzy3L,Liquid Blue,24,2015-08-07,1,2015
4,008rk8F6ZxspZT4bUlkIQG,Fever,38,2018-11-16,1,2018


In [24]:
# Playlists per track, and the check that it counts distinct playlists.
display(track_summary[["track_id", "playlist_count"]].head())
assert track_summary["track_id"].is_unique
assert track_summary["playlist_count"].ge(1).all()
print(f"Unique tracks: {track_summary['track_id'].nunique():,}")
print(f"Summary rows: {len(track_summary):,}")

,track_id,playlist_count
0,0017A6SJgTbfQVU2EtsPNo,1
1,002xjHwzEx66OWFV2IP9dk,1
2,004s3t0ONYlzxII9PLgU6z,1
3,008MceT31RotUANsKuzy3L,1
4,008rk8F6ZxspZT4bUlkIQG,1


Unique tracks: 28,356
Summary rows: 28,356


In [25]:
# Missingness in the variables used below.
used_columns = [
    "track_id", "playlist_id", "track_name",
    "track_popularity", "track_album_release_date"
 ]
missing_values = songs_raw[used_columns].isna().sum()
display(missing_values[missing_values > 0])
print("Release-date string lengths:")
display(
    songs_raw["track_album_release_date"]
    .astype("string")
    .str.len()
    .value_counts(dropna=False)
    .sort_index()
 )

track_name    5
dtype: int64

Release-date string lengths:


track_album_release_date
4      1855
7        31
10    30947
Name: count, dtype: Int64

In [26]:
# A release year we can defend for later comparisons.
invalid_years = track_summary["release_year"].isna().sum()
print(f"Missing or unparseable release years: {invalid_years:,}")
display(
    track_summary[["release_date", "release_year"]]
    .drop_duplicates()
    .sort_values("release_date")
    .head(10)
 )

Missing or unparseable release years: 0


,release_date,release_year
26152,1957-01-01,1957
24864,1957-03,1957
17027,1958-03-21,1958
745,1960,1960
118,1961-10-26,1961
16942,1962,1962
14900,1963,1963
12120,1963-03-22,1963
27597,1963-05-27,1963
3323,1963-07-08,1963


### Question 1 answers

**One row of the raw file is:** one track appearing in one playlist, with playlist, genre, and audio-feature information.

**One row of our working table is:** one unique `track_id`, with representative track information and the number of distinct playlists containing it. That grain is chosen because the question asks how far each track travels.

**counted playlists per track using:** `groupby("track_id")` and `nunique("playlist_id")`. **The check that convinces it counts playlists:** the summary has 28,356 unique track IDs and the example track has two raw playlist associations and a count of two.

**Missing values in the columns used:** five track names are missing; the identifiers, popularity values, playlist IDs, and release dates needed for the main comparison are available. **What is done about each:** kept missing names because names are labels, and no replace missing values with invented values.

**The release dates record mixed precision, so the year derived is** the first four characters of the observed date. **It would have been invented by parsing them all to a full day is** a month and day for entries that only record a year or year-month.

## Question 2 · Attach the family label

Write your predicted row count down before you merge.


In [27]:
# Audit the lookup and predict the row count before merging.
family_key_counts = families_raw.groupby("subgenre")["family"].nunique()
ambiguous_subgenres = set(
    family_key_counts[family_key_counts > 1].index
 )
family_one_to_one = families_raw[
    ~families_raw["subgenre"].isin(ambiguous_subgenres)
].copy()

track_subgenres = songs_raw[["track_id", "playlist_subgenre"]].drop_duplicates()
track_subgenres = track_subgenres.rename(columns={"playlist_subgenre": "subgenre"})
eligible_associations = track_subgenres[
    track_subgenres["subgenre"].isin(family_one_to_one["subgenre"])
].copy()
expected_rows = len(eligible_associations)
print(f"Predicted rows after a one-to-one inner merge: {expected_rows:,}")

Predicted rows after a one-to-one inner merge: 28,610


In [28]:
# Keep the merge auditable: validate the lookup and mark matches.
family_merge = track_subgenres.merge(
    family_one_to_one,
    on="subgenre",
    how="left",
    validate="many_to_one",
    indicator=True
 )
display(family_merge["_merge"].value_counts())
display(family_merge.head())

_merge
both          28610
left_only      4223
right_only        0
Name: count, dtype: int64

,track_id,subgenre,family,_merge
0,6f807x0ima9a1j3VPbc7VN,dance pop,NaN,left_only
1,0r7CVbZTWZgbTCYdfa2P31,dance pop,NaN,left_only
2,1z1Hg7Vb0AhHDiEmnDE79l,dance pop,NaN,left_only
3,75FpbthrwQmzHlBJLuGdC7,dance pop,NaN,left_only
4,1e8PAfcKUYoKkxPhrHqw4x,dance pop,NaN,left_only


In [29]:
# Identify ambiguous and unmapped subgenre keys.
observed_subgenres = set(track_subgenres["subgenre"].dropna())
mapped_subgenres = set(family_one_to_one["subgenre"].dropna())
unmapped_subgenres = sorted(observed_subgenres - mapped_subgenres)
print(f"Ambiguous subgenres: {sorted(ambiguous_subgenres)}")
print(f"Unmapped subgenres: {unmapped_subgenres}")
display(
    family_key_counts[family_key_counts > 1]
    .rename("number_of_family_values")
    .to_frame()
 )

Ambiguous subgenres: ['dance pop']
Unmapped subgenres: ['dance pop', 'neo soul', 'tropical']


,number_of_family_values
subgenre,
dance pop,2


In [30]:
# Keep valid one-to-one matches and record the cost of the decision.
family_associations = family_merge.loc[
    family_merge["_merge"] == "both",
    ["track_id", "subgenre", "family"]
].copy()
unmatched_associations = family_merge.loc[
    family_merge["_merge"] == "left_only"
].copy()
assert len(family_associations) == expected_rows
assert family_associations["subgenre"].isin(ambiguous_subgenres).sum() == 0
print(f"Valid family associations kept: {len(family_associations):,}")
print(f"Associations without a family label: {len(unmatched_associations):,}")
print("Decision: exclude ambiguous and unmapped associations rather than invent a family label.")

Valid family associations kept: 28,610
Associations without a family label: 4,223
Decision: exclude ambiguous and unmapped associations rather than invent a family label.


### Question 2 answers

**Rows predicted after the merge:** 28,610 valid track-subgenre associations. **Rows got:** 28,610 matched associations; the left merge also identified 4,223 unmatched associations.

**What is wrong with the keys:** `dance pop` maps to two families, so it is ambiguous. `neo soul` and `tropical` occur in the song data but are not present in the family lookup.

**What decided about each problem, and what it cost:** removed the ambiguous mapping before the merge and kept only valid one-to-one matches. This avoids duplicated or invented family assignments, but it excludes some associations from family-level comparisons.

**Rows with no family label at the end:** 4,223. **What is done with them:** they are retained in the audit output but excluded them from family-level summaries.